In [ ]:
# Purpose: Export processed outputs for downstream use.
# Notes: Documentation-only update; logic and execution order are unchanged.

from pathlib import Path
from bs4 import BeautifulSoup
from openai import OpenAI
from difflib import SequenceMatcher
from rapidfuzz import process, fuzz

import json
import os
import re
import time
import pandas as pd
import requests
import unicodedata

# ------------------------------
# Reusable project path setup
# ------------------------------
USERNAME = "gebruiker"
PROJECT_FOLDER_NAME = "MEP"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if p.name == PROJECT_FOLDER_NAME),
    NOTEBOOK_DIR.parent,
 )

WIKIPEDIA_DIR = PROJECT_ROOT / "4 Wikipedia"
LDA_DIR = PROJECT_ROOT / "4 LDA Topic Analysis"
LLM_WIKIPEDIA_DIR = PROJECT_ROOT / "5 LLM Wikipedia analysis"

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = LLM_WIKIPEDIA_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES = [
    WIKIPEDIA_DIR / "outputs" / "tables" / "mep_wikipedia_final.xlsx",
    WIKIPEDIA_DIR / "mep_wikipedia_final.xlsx",
]
POLARISATION_CANDIDATES = [
    LDA_DIR / "outputs" / "tables" / "polarisation_scores_scored_only.csv",
    LDA_DIR / "polarisation_scores_scored_only.csv",
]

INPUT_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), INPUT_CANDIDATES[0])
POLARISATION_SCORED_ONLY_PATH = next((p for p in POLARISATION_CANDIDATES if p.exists()), POLARISATION_CANDIDATES[0])

OUTPUT_PATH = TABLES_DIR / "wikipedia_extracted.xlsx"
PARTIAL_PATH = TABLES_DIR / "wikipedia_extracted_partial.xlsx"
FINAL_POLARISATION_PATH = TABLES_DIR / "extracted_polarisation_final.xlsx"
AUDIT_OUTPUT_PATH = TABLES_DIR / "polarisation_vs_extracted_match_audit.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input file:   {INPUT_PATH}")
print(f"Output file:  {OUTPUT_PATH}")



# Wikipedia Biographical Extraction (LLM)

This notebook extracts structured biographical information from Wikipedia pages and creates a final table for scored MEPs.

Run from top to bottom. Paths are configured once in the next cell for portability and easier GitHub reuse.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

# -------------------
# Main extraction configuration and shared helpers
# -------------------
N_FIRST = 1919  # limit rows for test runs if needed

BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "meta-llama/llama-4-maverick:free"

# Do not hardcode secrets in notebooks.
API_KEY = (
    (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    or (os.environ.get("OPENROUTER_KEY") or "").strip()
 )

if not API_KEY:
    print("OPENROUTER_API_KEY not found. Set it before running extraction cells.")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

SYSTEM = (
    "You extract and classify structured biographical info from Wikipedia text. "
    "Return only valid JSON that matches the requested schema exactly. "
    "Use ONLY the allowed bucket values. "
    "If unknown / not stated in the text, use null (or empty list where applicable). "
    "Do not invent details. "
    "For classifications, choose the closest bucket only when the text supports it. "
    "Always provide short evidence snippets copied from the text (max ~200 chars each)."
 )

ALLOWED = {
  "gender_bucket": ["Male", "Female", "Other or Not stated"],
  "region_bucket": ["Northern Europe", "Western Europe", "Southern Europe", "Central & Eastern Europe", "Baltic"],
  "education_level_bucket": ["No higher education listed", "Bachelor", "Master", "Doctorate"],
  "education_field_bucket": [
      "Economics or Business", "Law", "Political Science or Public Administration",
      "Sociology or Demography or Social Sciences", "Humanities", "STEM", "Medicine or Health",
      "Education or Teaching", "Other or Unclear",
  ],
  "professional_background_bucket": [
      "Business or Finance", "Law", "Academia or Research", "Public Administration", "NGO or Civil Society",
      "Trade Union", "Journalism or Media", "Education Sector", "Health Sector", "Manual or Technical",
      "Career Politician", "Other or Unclear",
  ],
  "religion_bucket": ["Christian", "Other religion", "Explicitly secular", "Not mentioned"],
  "parental_status_bucket": ["Has children", "No children", "Not mentioned"],
  "migration_background_bucket": ["Born abroad", "Parent foreign-born", "None mentioned"],
  "international_experience_bucket": ["Studied abroad", "Worked abroad", "EU institutional career", "None significant mentioned"],
  "military_background_bucket": ["Yes", "No", "Not mentioned"],
  "prior_political_level_bucket": ["Local only", "National parliament", "Government minister", "No prior elected office", "Unclear"],
  "demography_expertise_bucket": ["Demography specialist", "Social policy background", "None"],
}

def html_to_text(html: str) -> str:
    if not isinstance(html, str) or not html.strip():
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def safe_json_loads(s: str) -> dict:
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    match = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if match:
        s = match.group(0).strip()

    try:
        return json.loads(s)
    except json.JSONDecodeError:
        s2 = re.sub(r",\s*([}\]])", r"\1", s)
        s2 = s2.replace("\u00a0", " ").strip()
        return json.loads(s2)

def is_mep_from_text(text: str) -> bool:
    if not text:
        return False
    patterns = [
        r"\bMember of the European Parliament\b",
        r"\bMEP\b",
        r"\bMembers of the European Parliament\b",
        r"\bEuropean Parliament\b.*\bmember\b",
        r"\b(member|deputy)\b.*\bEuropean Parliament\b",
    ]
    text_flat = " ".join(text.split())
    return any(re.search(p, text_flat, flags=re.IGNORECASE) for p in patterns)

def flatten_extract(d: dict) -> dict:
    out = {}
    out["full_name_extracted"] = d.get("full_name")
    out["birth_date_extracted"] = d.get("birth_date")
    out["place_of_birth_extracted"] = d.get("place_of_birth")
    out["country_of_origin_extracted"] = d.get("country_of_origin")
    out["ep_group_extracted"] = d.get("ep_group")
    out["mep_since_extracted"] = d.get("mep_since")
    out["occupations_extracted"] = json.dumps(d.get("occupations", []), ensure_ascii=False)
    out["parties_extracted"] = json.dumps(d.get("parties", []), ensure_ascii=False)
    out["education_extracted"] = json.dumps(d.get("education", []), ensure_ascii=False)
    out["confidence_extracted"] = d.get("confidence")

    buckets = d.get("buckets", {}) or {}
    for key, value in buckets.items():
        out[key] = value

    evidence = d.get("evidence", {}) or {}
    for key, value in evidence.items():
        out[f"evidence_{key}"] = value

    return out


In [3]:
# Purpose: Transform and aggregate data into analysis-ready structures.
# Notes: Documentation-only update; logic and execution order are unchanged.

# Prefer known working free models first; dynamic discovery appends more.
STATIC_MODEL_CANDIDATES = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "openai/gpt-oss-20b:free",
    "qwen/qwen3-coder:free",
    "google/gemma-4-31b-it:free",
    "meta-llama/llama-3.2-3b-instruct:free",
]

# Keep requests small enough for free-tier limits.
MAX_INPUT_CHARS = 18000
MAX_OUTPUT_TOKENS = 1200
MAX_RETRIES_PER_MODEL = 2
BACKOFF_SECONDS = 1.5
LIVE_FREE_LIMIT = 30

LAST_MODEL_USED = None

def _discover_live_free_models(limit: int = LIVE_FREE_LIMIT) -> list[str]:
    """Fetch currently live free model slugs from OpenRouter."""
    try:
        resp = requests.get(
            f"{BASE_URL}/models",
            headers={"Authorization": f"Bearer {API_KEY}"},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json().get("data", [])
        ids = [m.get("id", "") for m in data if isinstance(m, dict)]
        ids = [m for m in ids if m.endswith(":free")]
        return ids[:limit]
    except Exception:
        return []

def _build_model_candidates() -> list[str]:
    live = _discover_live_free_models()
    merged = []
    for m in STATIC_MODEL_CANDIDATES + live:
        m = (m or "").strip()
        if m and m not in merged:
            merged.append(m)
    return merged

MODEL_CANDIDATES = _build_model_candidates()

def _is_non_retriable_model_error(err_text: str) -> bool:
    t = err_text.lower()
    return (
        "error code: 404" in t
        or "no endpoints found" in t
        or "unavailable for free" in t
        or "model not found" in t
    )

def _build_extraction_prompt(raw_text: str, name_hint: str, lang: str) -> str:
    return f"""
Name hint: {name_hint}
Language of page: {lang}

Task:
Return ONLY valid JSON matching the schema. No commentary.

Allowed buckets (exact strings):
{json.dumps(ALLOWED, ensure_ascii=False, indent=2)}

JSON schema:
{{
  "full_name": string|null,
  "birth_date": string|null,
  "place_of_birth": string|null,
  "country_of_origin": string|null,
  "education": [{{"degree": string|null,"field": string|null,"institution": string|null,"country": string|null,"start_year": integer|null,"end_year": integer|null,"notes": string|null}}],
  "occupations": [string],
  "parties": [string],
  "ep_group": string|null,
  "mep_since": string|null,
  "buckets": {{
    "gender_bucket": string|null,
    "region_bucket": string|null,
    "education_level_bucket": string|null,
    "education_field_bucket": string|null,
    "professional_background_bucket": string|null,
    "religion_bucket": string|null,
    "parental_status_bucket": string|null,
    "migration_background_bucket": string|null,
    "international_experience_bucket": string|null,
    "military_background_bucket": string|null,
    "prior_political_level_bucket": string|null,
    "demography_expertise_bucket": string|null
  }},
  "confidence": number,
  "evidence": {{
    "birth_date": string|null,
    "country_of_origin": string|null,
    "ep_group": string|null,
    "education_level_bucket": string|null,
    "education_field_bucket": string|null,
    "professional_background_bucket": string|null,
    "parental_status_bucket": string|null,
    "migration_background_bucket": string|null,
    "international_experience_bucket": string|null,
    "prior_political_level_bucket": string|null
  }}
}}

Wikipedia text:
\"\"\"{raw_text[:MAX_INPUT_CHARS]}\"\"\"
""".strip()

def _call_model(model_name: str, prompt: str):
    return client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=MAX_OUTPUT_TOKENS,
    )

def extract_with_llm(raw_text: str, name_hint: str, lang: str) -> dict:
    global LAST_MODEL_USED
    prompt = _build_extraction_prompt(raw_text, name_hint, lang)
    errors = []

    if not MODEL_CANDIDATES:
        raise RuntimeError("No free model candidates available right now.")

    for model_name in MODEL_CANDIDATES:
        for attempt in range(1, MAX_RETRIES_PER_MODEL + 1):
            try:
                resp = _call_model(model_name, prompt)
                content = (resp.choices[0].message.content or "").strip()
                if not content:
                    raise ValueError("Empty model output")

                try:
                    parsed = safe_json_loads(content)
                    LAST_MODEL_USED = model_name
                    return parsed
                except Exception:
                    fixer = f"""
Fix the following so it is strictly valid JSON.
Do not change meaning or values.
Return JSON only.

INPUT:
{content}
""".strip()

                    resp2 = client.chat.completions.create(
                        model=model_name,
                        messages=[
                            {"role": "system", "content": "Return only valid JSON."},
                            {"role": "user", "content": fixer},
                        ],
                        temperature=0.0,
                        max_tokens=MAX_OUTPUT_TOKENS,
                    )
                    fixed = (resp2.choices[0].message.content or "").strip()
                    parsed = safe_json_loads(fixed)
                    LAST_MODEL_USED = model_name
                    return parsed

            except Exception as e:
                msg = str(e)
                errors.append(f"{model_name} attempt {attempt}: {msg}")
                if _is_non_retriable_model_error(msg):
                    break
                time.sleep(BACKOFF_SECONDS * attempt)

    short_errors = "\n".join(errors[-8:])
    raise RuntimeError(
        "All free model attempts failed. Last errors:\n" + short_errors
    )


In [4]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# -------------------
# RUN
# -------------------
df = pd.read_excel(INPUT_PATH)
df_first = df.head(N_FIRST).copy()

rows = []
for idx, row in df_first.iterrows():
    name_hint = str(row.get("name", "") or "")
    lang = str(row.get("wikipedia_language", "") or "")

    html_en = row.get("wikipedia_content_html_en")
    html_local = row.get("wikipedia_content_html")

    text_en = html_to_text(html_en) if isinstance(html_en, str) else ""
    text_local = html_to_text(html_local) if isinstance(html_local, str) else ""

    # combine
    text = text_en + "\n\n" + text_local


    extracted = extract_with_llm(text, name_hint=name_hint, lang=lang)
    flat = flatten_extract(extracted)

    # MEP flag: combine LLM fields + simple text heuristic
    mep_flag = bool(extracted.get("mep_since") or extracted.get("ep_group")) or is_mep_from_text(text)
    flat["MEP"] = bool(mep_flag)

    rows.append(flat)

df_out = pd.concat([df_first.reset_index(drop=True), pd.DataFrame(rows)], axis=1)
df_out.to_excel(OUTPUT_PATH, index=False)

print(f"Wrote: {OUTPUT_PATH}")



KeyboardInterrupt: 

In [ ]:
# Purpose: Export processed outputs for downstream use.
# Notes: Documentation-only update; logic and execution order are unchanged.

partial_out = pd.concat([df_first.iloc[:len(rows)].reset_index(drop=True), pd.DataFrame(rows)], axis=1)
partial_out.to_excel(PARTIAL_PATH, index=False)
print(f"Wrote partial: {PARTIAL_PATH} ({len(rows)} rows)")


Wrote partial: C:\Users\Gebruiker\OneDrive - UniversitÃƒÂ  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\5 LLM Wikipedia analysis\wikipedia_extracted_partial.xlsx (554 rows)


## Incremental extraction for polarisation-scored MEPs

This section builds extracted_polarisation_final.xlsx by:
1. Loading polarisation_scores_scored_only.csv.
2. Reusing already-processed MEPs from the partial extraction file.
3. Running extraction only for missing scored MEPs.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ------------------------------------------------------------
# Incremental run for scored MEPs only (score-first final merge)
# ------------------------------------------------------------

try:
    HAS_RAPIDFUZZ = True
except Exception:
    HAS_RAPIDFUZZ = False

# Match thresholds for rapidfuzz scorers (0-100).
FUZZY_TOKEN_SET_THRESHOLD = 72
FUZZY_TOKEN_SORT_THRESHOLD = 70
FUZZY_PARTIAL_THRESHOLD = 82

RUN_EXTRACTION = True
CHECKPOINT_EVERY = 1

def _norm_name(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def _match_key(x: str) -> str:
    stop_tokens = {"mr", "mrs", "ms", "dr", "prof", "sir", "jr", "sr", "mep"}
    tokens = [t for t in _norm_name(x).split() if t and t not in stop_tokens]
    return " ".join(sorted(tokens))

def _best_fuzzy_match(name_key: str, candidates: list[str]) -> tuple[str | None, float, str]:
    if not name_key or not candidates:
        return None, 0.0, "none"

    if HAS_RAPIDFUZZ:
        best_name, best_score, best_method = None, 0.0, "none"
        for scorer_name, scorer, threshold in [
            ("token_set", fuzz.token_set_ratio, FUZZY_TOKEN_SET_THRESHOLD),
            ("token_sort", fuzz.token_sort_ratio, FUZZY_TOKEN_SORT_THRESHOLD),
            ("partial", fuzz.partial_ratio, FUZZY_PARTIAL_THRESHOLD),
        ]:
            res = process.extractOne(name_key, candidates, scorer=scorer, score_cutoff=threshold)
            if res:
                cand, score, _ = res
                if float(score) > best_score:
                    best_name, best_score, best_method = cand, float(score), scorer_name

        if best_name is not None:
            return best_name, best_score, best_method

        res_any = process.extractOne(name_key, candidates, scorer=fuzz.token_set_ratio)
        if res_any:
            cand, score, _ = res_any
            return cand, float(score), "token_set_below_threshold"

        return None, 0.0, "none"

    best_name, best_score = None, 0.0
    for candidate in candidates:
        score = SequenceMatcher(None, name_key, candidate).ratio() * 100.0
        if score > best_score:
            best_name, best_score = candidate, score
    return best_name, best_score, "difflib"

def _build_extracted_df(existing_df: pd.DataFrame, new_rows_list: list, target_keys: set) -> pd.DataFrame:
    parts_local = []
    if not existing_df.empty:
        tmp_existing = existing_df.copy()
        tmp_existing["_target_name_key"] = tmp_existing["_name_key"]
        tmp_existing["name_match_type"] = tmp_existing.get("name_match_type", "partial_exact")
        tmp_existing["name_match_score"] = tmp_existing.get("name_match_score", 100.0)
        parts_local.append(tmp_existing)
    if new_rows_list:
        parts_local.append(pd.DataFrame(new_rows_list))

    if parts_local:
        out = pd.concat(parts_local, ignore_index=True)
        out["_name_key"] = out["_target_name_key"].fillna("")
        out = out[out["_name_key"].isin(target_keys)].copy()
        out = out.drop_duplicates(subset=["_name_key"], keep="first")
        return out

    return pd.DataFrame(columns=["_name_key"])

def _write_checkpoint(pol_base: pd.DataFrame, existing_df: pd.DataFrame, new_rows_list: list, target_keys: set, out_path: Path) -> int:
    extracted_tmp = _build_extracted_df(existing_df, new_rows_list, target_keys)
    final_tmp = pol_base.merge(extracted_tmp, on="_name_key", how="left", suffixes=("", "_wiki"))
    final_tmp = final_tmp.drop(columns=["_name_key", "_name_norm"], errors="ignore")
    final_tmp.to_excel(out_path, index=False)
    return len(final_tmp)

pol_df = pd.read_csv(POLARISATION_SCORED_ONLY_PATH).copy()
pol_name_candidates = ["mep_name_raw", "mep_name_clean", "name"]
pol_name_col = next((c for c in pol_name_candidates if c in pol_df.columns), None)
if pol_name_col is None:
    raise ValueError(
        f"Could not find a name column in polarisation file. Columns: {list(pol_df.columns)}"
    )

pol_df["_name_norm"] = pol_df[pol_name_col].fillna("").astype(str).map(_norm_name)
pol_df["_name_key"] = pol_df[pol_name_col].fillna("").astype(str).map(_match_key)
pol_df = pol_df[pol_df["_name_key"] != ""].copy()
pol_df = pol_df.drop_duplicates(subset=["_name_key"], keep="first").reset_index(drop=True)
target_name_keys = set(pol_df["_name_key"].tolist())

if PARTIAL_PATH.exists():
    try:
        partial_df = pd.read_excel(PARTIAL_PATH).copy()
    except BaseException as e:
        print(f"Could not load partial file quickly ({e}). Continuing without partial reuse.")
        partial_df = pd.DataFrame()
else:
    partial_df = pd.DataFrame()

if "name" in partial_df.columns:
    partial_name_col = "name"
elif "full_name_extracted" in partial_df.columns:
    partial_name_col = "full_name_extracted"
else:
    partial_name_col = None

if partial_name_col is not None and not partial_df.empty:
    partial_df["_name_norm"] = partial_df[partial_name_col].fillna("").astype(str).map(_norm_name)
    partial_df["_name_key"] = partial_df[partial_name_col].fillna("").astype(str).map(_match_key)
    existing_extracted_df = partial_df[partial_df["_name_key"].isin(target_name_keys)].copy()
    existing_extracted_df = existing_extracted_df.drop_duplicates(subset=["_name_key"], keep="first")
else:
    existing_extracted_df = pd.DataFrame()

processed_name_keys = set(existing_extracted_df.get("_name_key", pd.Series(dtype=str)).tolist())
missing_name_keys = [k for k in pol_df["_name_key"].tolist() if k not in processed_name_keys]

print("Rapidfuzz available:", HAS_RAPIDFUZZ)
print("Polarisation-scored targets:", len(target_name_keys))
print("Already extracted from partial:", len(processed_name_keys))
print("Need new extraction:", len(missing_name_keys))

source_df = pd.read_excel(INPUT_PATH).copy()
if "name" not in source_df.columns:
    raise ValueError("INPUT_PATH must contain a 'name' column for matching.")
source_df["_name_norm"] = source_df["name"].fillna("").astype(str).map(_norm_name)
source_df["_name_key"] = source_df["name"].fillna("").astype(str).map(_match_key)
source_df = source_df[source_df["_name_key"] != ""].copy()
source_df = source_df.drop_duplicates(subset=["_name_key"], keep="first").reset_index(drop=True)

source_key_set = set(source_df["_name_key"].tolist())
source_key_list = source_df["_name_key"].tolist()

mapped_rows = []
exact_matches, fuzzy_matches, unmatched = 0, 0, 0

for tkey in missing_name_keys:
    if tkey in source_key_set:
        mapped_rows.append({
            "_target_name_key": tkey,
            "_source_name_key": tkey,
            "match_type": "exact",
            "match_score": 100.0,
        })
        exact_matches += 1
    else:
        best_key, best_score, method = _best_fuzzy_match(tkey, source_key_list)
        if best_key is not None and method not in {"none", "token_set_below_threshold"}:
            mapped_rows.append({
                "_target_name_key": tkey,
                "_source_name_key": best_key,
                "match_type": f"fuzzy_{method}",
                "match_score": float(best_score),
            })
            fuzzy_matches += 1
        else:
            unmatched += 1
            if best_key is not None:
                mapped_rows.append({
                    "_target_name_key": tkey,
                    "_source_name_key": best_key,
                    "match_type": "below_threshold",
                    "match_score": float(best_score),
                })

map_df = pd.DataFrame(mapped_rows)
if not map_df.empty:
    map_df = map_df.drop_duplicates(subset=["_target_name_key"], keep="first")

print("Name mapping stats -> exact:", exact_matches, "fuzzy:", fuzzy_matches, "unmatched:", unmatched)

if not map_df.empty:
    low = map_df[map_df["match_type"].isin(["below_threshold"])]
    if not low.empty:
        print("Sample below-threshold matches (top 10):")
        display(low.sort_values("match_score", ascending=False).head(10))

if not map_df.empty:
    todo_df = map_df[map_df["match_type"] != "below_threshold"].merge(
        source_df,
        left_on="_source_name_key",
        right_on="_name_key",
        how="left",
    )
else:
    todo_df = pd.DataFrame()

print("Matched source rows for missing names:", len(todo_df))

new_rows = []
interrupted = False
if RUN_EXTRACTION:
    total = len(todo_df)
    try:
        for i, (_, row) in enumerate(todo_df.iterrows(), start=1):
            name_hint = str(row.get("name", "") or "")
            target_key = str(row.get("_target_name_key", "") or "")
            print(f"[{i}/{total}] Working on: {name_hint} (target_key={target_key})")

            lang = str(row.get("wikipedia_language", "") or "")
            html_en = row.get("wikipedia_content_html_en")
            html_local = row.get("wikipedia_content_html")

            text_en = html_to_text(html_en) if isinstance(html_en, str) else ""
            text_local = html_to_text(html_local) if isinstance(html_local, str) else ""
            text = text_en + "\n\n" + text_local

            try:
                extracted = extract_with_llm(text, name_hint=name_hint, lang=lang)
                flat = flatten_extract(extracted)
                mep_flag = bool(extracted.get("mep_since") or extracted.get("ep_group")) or is_mep_from_text(text)
                flat["MEP"] = bool(mep_flag)
                flat["extract_error"] = pd.NA
            except Exception as e:
                flat = {"MEP": pd.NA, "extract_error": str(e)}

            flat["_target_name_key"] = target_key
            flat["_name_key"] = target_key
            flat["name_match_type"] = row.get("match_type", pd.NA)
            flat["name_match_score"] = row.get("match_score", pd.NA)

            merged_row = {**row.to_dict(), **flat}
            new_rows.append(merged_row)

            if CHECKPOINT_EVERY > 0 and (i % CHECKPOINT_EVERY == 0 or i == total):
                written_rows = _write_checkpoint(
                    pol_base=pol_df,
                    existing_df=existing_extracted_df,
                    new_rows_list=new_rows,
                    target_keys=target_name_keys,
                    out_path=FINAL_POLARISATION_PATH,
                )
                print(f"Checkpoint saved at {i}/{total}. Rows currently written: {written_rows}")
    except KeyboardInterrupt:
        interrupted = True
        print("Interrupted by user. Saving collected progress...")
        written_rows = _write_checkpoint(
            pol_base=pol_df,
            existing_df=existing_extracted_df,
            new_rows_list=new_rows,
            target_keys=target_name_keys,
            out_path=FINAL_POLARISATION_PATH,
        )
        print(f"Checkpoint written after interrupt. Rows currently written: {written_rows}")
else:
    print("RUN_EXTRACTION=False -> mapping only, no LLM calls made.")

new_extracted_df = pd.DataFrame(new_rows)
extracted_df = _build_extracted_df(existing_extracted_df, new_rows, target_name_keys)

final_df = pol_df.merge(extracted_df, on="_name_key", how="left", suffixes=("", "_wiki"))
final_df = final_df.drop(columns=["_name_key", "_name_norm"], errors="ignore")
final_df.to_excel(FINAL_POLARISATION_PATH, index=False)

print("Newly extracted in this run:", len(new_extracted_df))
print("Final rows saved (should equal scored targets):", len(final_df))
if interrupted:
    print("Run ended by user interrupt after saving progress.")
print(f"Wrote: {FINAL_POLARISATION_PATH}")


Rapidfuzz available: True
Polarisation-scored targets: 158
Already extracted from partial: 75
Need new extraction: 83
Name mapping stats -> exact: 80 fuzzy: 3 unmatched: 0
Matched source rows for missing names: 83
[1/83] Working on: GIEREK Adam (target_key=adam gierek)
Checkpoint saved at 1/83. Rows currently written: 158
[2/83] Working on: SZEJNFELD Adam (target_key=adam szejnfeld)
Checkpoint saved at 2/83. Rows currently written: 158
[3/83] Working on: KOZÃ…ÂOWSKA-RAJEWICZ Agnieszka (target_key=agnieszka koz owska rajewicz)
Checkpoint saved at 3/83. Rows currently written: 158
[4/83] Working on: PATRICIELLO Aldo (target_key=aldo patriciello)
Checkpoint saved at 4/83. Rows currently written: 158
[5/83] Working on: HOMS GINEL Alicia (target_key=alicia ginel homs)
Checkpoint saved at 5/83. Rows currently written: 158
[6/83] Working on: PETERLE Alojz (target_key=alojz peterle)
Checkpoint saved at 6/83. Rows currently written: 158
[7/83] Working on: GOMES Ana (target_key=ana gomes)
Check

In [15]:
# Purpose: Export processed outputs for downstream use.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ------------------------------------------------------------
# Snapshot current in-memory progress after interrupt
# ------------------------------------------------------------

required_vars = [
    "new_rows",
    "pol_df",
    "existing_extracted_df",
    "target_name_keys",
    "FINAL_POLARISATION_PATH",
    "_build_extracted_df",
    "_write_checkpoint",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    print("Missing variables in memory:", missing)
    print("Run Cell 7 first to initialize extraction state.")
else:
    print("In-memory new rows collected this run:", len(new_rows))
    print("Rows already reused from partial:", len(existing_extracted_df))

    # Force-save checkpoint from current in-memory state.
    written_rows = _write_checkpoint(
        pol_base=pol_df,
        existing_df=existing_extracted_df,
        new_rows_list=new_rows,
        target_keys=target_name_keys,
        out_path=FINAL_POLARISATION_PATH,
    )
    print("Checkpoint written from memory. Total rows in final file:", written_rows)
    print(f"Checkpoint path: {FINAL_POLARISATION_PATH}")

    # Build extracted view from memory and report coverage.
    extracted_now = _build_extracted_df(existing_extracted_df, new_rows, target_name_keys)
    print("Extracted rows currently available in memory:", len(extracted_now))

    cols = [c for c in ["name", "_target_name_key", "name_match_type", "name_match_score", "extract_error", "full_name_extracted"] if c in pd.DataFrame(new_rows).columns]
    if len(new_rows) > 0 and cols:
        print("Sample of in-memory rows (last 10):")
        display(pd.DataFrame(new_rows)[cols].tail(10))


Missing variables in memory: ['_build_extracted_df', '_write_checkpoint']
Run Cell 7 first to initialize extraction state.


In [10]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ------------------------------------------------------------
# Add old extracted-file MEP names to final polarisation file
# ------------------------------------------------------------

def _norm_name_local(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

final_df = pd.read_excel(FINAL_POLARISATION_PATH)
partial_df = pd.read_excel(PARTIAL_PATH) if PARTIAL_PATH.exists() else pd.DataFrame()

# Determine name columns in final and partial files.
final_name_candidates = ["mep_name_raw", "mep_name_clean", "name"]
final_name_col = next((c for c in final_name_candidates if c in final_df.columns), None)
if final_name_col is None:
    raise ValueError(f"No name column found in final file: {list(final_df.columns)}")

partial_name_candidates = ["name", "full_name_extracted", "mep_name_raw", "mep_name_clean"]
partial_name_col = next((c for c in partial_name_candidates if c in partial_df.columns), None)
if partial_name_col is None:
    raise ValueError(
        "No suitable old-name column found in partial extracted file. "
        f"Columns: {list(partial_df.columns)}"
    )

final_df["_name_norm"] = final_df[final_name_col].fillna("").astype(str).map(_norm_name_local)
partial_df["_name_norm"] = partial_df[partial_name_col].fillna("").astype(str).map(_norm_name_local)

# Keep one old-name record per normalized name.
partial_names = (
    partial_df[["_name_norm", partial_name_col]]
    .dropna(subset=["_name_norm"])
    .drop_duplicates(subset=["_name_norm"], keep="first")
    .rename(columns={partial_name_col: "mep_name_old_from_extracted"})
 )

final_df = final_df.merge(partial_names, on="_name_norm", how="left")
final_df = final_df.drop(columns=["_name_norm"], errors="ignore")

final_df.to_excel(FINAL_POLARISATION_PATH, index=False)
print("Added old extracted names column: mep_name_old_from_extracted")
print(f"Wrote: {FINAL_POLARISATION_PATH}")


Added old extracted names column: mep_name_old_from_extracted
Wrote: C:\Users\Gebruiker\OneDrive - UniversitÃƒÂ  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\5 LLM Wikipedia analysis\extracted_polarisation_final.xlsx


## Match Audit: Polarisation-Scored MEPs vs Extracted Wikipedia MEPs

This check uses the polarisation-scored MEP list as the base and verifies which of those have a matched row in `extracted_polarisation_final.xlsx`.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
# Purpose: Load input data and initialize working tables for the next analysis steps.
# Notes: Documentation-only update; logic and execution order are unchanged.

# ------------------------------------------------------------
# Audit coverage from scored MEPs -> extracted final file
# ------------------------------------------------------------

def _norm_name_audit(x: str) -> str:
    if not isinstance(x, str):
        return ""
    x = "".join(
        c for c in unicodedata.normalize("NFKD", x)
        if not unicodedata.combining(c)
    )
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def _match_key_audit(x: str) -> str:
    stop_tokens = {"mr", "mrs", "ms", "dr", "prof", "sir", "jr", "sr", "mep"}
    tokens = [t for t in _norm_name_audit(x).split() if t and t not in stop_tokens]
    return " ".join(sorted(tokens))

scored_df = pd.read_csv(POLARISATION_SCORED_ONLY_PATH).copy()
scored_name_candidates = ["mep_name_raw", "mep_name_clean", "name"]
scored_name_col = next((c for c in scored_name_candidates if c in scored_df.columns), None)
if scored_name_col is None:
    raise ValueError(f"No name column found in scored file. Columns: {list(scored_df.columns)}")

scored_df["_name_key"] = scored_df[scored_name_col].fillna("").astype(str).map(_match_key_audit)
scored_df = scored_df[scored_df["_name_key"] != ""].copy()
scored_df = scored_df.drop_duplicates(subset=["_name_key"], keep="first").reset_index(drop=True)

if not FINAL_POLARISATION_PATH.exists():
    raise FileNotFoundError(f"Missing file: {FINAL_POLARISATION_PATH}")

final_df = pd.read_excel(FINAL_POLARISATION_PATH).copy()
final_name_candidates = ["mep_name_raw", "mep_name_clean", "name", "full_name_extracted"]
final_name_col = next((c for c in final_name_candidates if c in final_df.columns), None)
if final_name_col is None:
    raise ValueError(f"No name column found in final file. Columns: {list(final_df.columns)}")

if "_name_key" in final_df.columns:
    final_df["_name_key"] = final_df["_name_key"].fillna("").astype(str)
else:
    final_df["_name_key"] = final_df[final_name_col].fillna("").astype(str).map(_match_key_audit)

extracted_signal_cols = [
    "full_name_extracted",
    "birth_date_extracted",
    "country_of_origin_extracted",
    "education_extracted",
    "ep_group_extracted",
    "confidence_extracted",
]
present_signal_cols = [c for c in extracted_signal_cols if c in final_df.columns]
if not present_signal_cols:
    raise ValueError(
        "Could not find extracted signal columns in final file. "
        f"Expected any of: {extracted_signal_cols}"
    )

final_df["has_wikipedia_match"] = final_df[present_signal_cols].notna().any(axis=1)
match_lookup = final_df.groupby("_name_key", dropna=False)["has_wikipedia_match"].max().to_dict()

audit_df = scored_df[[scored_name_col, "_name_key"]].copy()
audit_df["has_wikipedia_match"] = audit_df["_name_key"].map(match_lookup).fillna(False)

matched_count = int(audit_df["has_wikipedia_match"].sum())
total_count = int(len(audit_df))
unmatched_count = total_count - matched_count

print(f"Scored MEPs (base): {total_count}")
print(f"Scored MEPs with Wikipedia-extracted match: {matched_count}")
print(f"Scored MEPs without match: {unmatched_count}")

unmatched_df = audit_df[~audit_df["has_wikipedia_match"]].copy()
if unmatched_df.empty:
    print("All scored MEPs have a Wikipedia-extracted match.")
else:
    print("Sample unmatched scored MEPs (first 20):")
    display(unmatched_df.head(20))

audit_df.to_csv(AUDIT_OUTPUT_PATH, index=False)
print(f"Wrote audit file: {AUDIT_OUTPUT_PATH}")


Scored MEPs (base): 158
Scored MEPs with Wikipedia-extracted match: 158
Scored MEPs without match: 0
All scored MEPs have a Wikipedia-extracted match.
Wrote audit file: C:\Users\Gebruiker\OneDrive - UniversitÃƒÂ  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\5 LLM Wikipedia analysis\polarisation_vs_extracted_match_audit.csv


## Final manual adjustments

After the automated extraction workflow, I manually reviewed and corrected a small number of country irregularities.

The final hand-in file is:
- extracted_polarisation_final_manually_improved_1906.xlsx

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.
